# Group 32 — Explainable JD–Resume Matching: Pipeline, Benchmarks and Evaluation

**Part I** is our first end-to-end prototype (one JD, one resume) and serves as the **baseline system**.
**Part II** turns it into an evaluated NLP study: an annotated dataset, alternative methods for every
component, quantitative comparison, ablations, fairness/robustness tests and a demo.

| Member | Module | Research question | Main metrics |
|---|---|---|---|
| A | Data & annotation | Can humans agree on skills / priorities / fit? | Cohen's κ, Fleiss' κ |
| B | Skill extraction | Taxonomy NER vs. dictionary vs. LLM vs. fine-tuned NER | P / R / F1 (strict, soft) |
| C | Requirement classification | Rules vs. TF-IDF+LR vs. zero-shot NLI vs. DistilBERT | Accuracy, macro-F1 |
| D | Matching models | Lexical vs. bi-encoder vs. cross-encoder vs. our pipeline | Spearman ρ, AUC, macro-F1, NDCG@k |
| E | Scoring & ablation | Are hand-set thresholds/weights good? Which component matters? | Spearman ρ, Δ ablation |
| F | Fairness, robustness, errors, demo | Does the score change with names/wording? Where does it fail? | Δ score, error taxonomy |

# Part I — Baseline prototype (original notebook)

> **Read me first.** Part I is kept unchanged as the record of our first prototype and is the *baseline* in every
> comparison. It is **superseded by the modular code in `src/`** used in Part II. Some functions exist in both places
> with the same name but different behaviour (e.g. `clean_text`, `extract_education`, `classify_requirement`):
> Part II imports the `src/` versions explicitly, so the two never mix. For the current system, read `src/pipeline.py`.

# Step 1 Get JD and Resume

In [ ]:
from pathlib import Path

jd_path = Path("JD_example1_ntu_AI.txt")

jd_text = jd_path.read_text(encoding="utf-8")

print(jd_text)

In [ ]:
from pathlib import Path
from pypdf import PdfReader
from docx import Document

def read_resume(file_path):
    file_path = Path(file_path)
    file_type = file_path.suffix.lower()

    if file_type == ".txt":
        return file_path.read_text(encoding="utf-8")

    elif file_type == ".pdf":
        reader = PdfReader(file_path)
        pages = []

        for page in reader.pages:
            text = page.extract_text()
            if text:
                pages.append(text)

        return "\n".join(pages)

    elif file_type == ".docx":
        doc = Document(file_path)
        paragraphs = [p.text for p in doc.paragraphs if p.text.strip()]

        # Extract text from tables as well
        for table in doc.tables:
            for row in table.rows:
                for cell in row.cells:
                    if cell.text.strip():
                        paragraphs.append(cell.text)

        return "\n".join(paragraphs)

    else:
        raise ValueError("Unsupported file format. Use TXT, PDF, or DOCX.")

resume_path = "Resume2.pdf"  # Change to your resume file path

resume_text = read_resume(resume_path)

print(resume_text)

# Step 2 Text Preprocessing

In [ ]:
import re
import unicodedata

def clean_text(text):
    """
    Clean JD or resume text while preserving meaningful information.
    """

    if not isinstance(text, str):
        raise TypeError("Input must be a string.")

    # 1. Normalize Unicode characters
    text = unicodedata.normalize("NFKC", text)

    # 2. Remove null characters
    text = text.replace("\x00", "")

    # 3. Normalize line endings
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # 4. Replace tabs and repeated spaces with a single space
    text = re.sub(r"[ \t]+", " ", text)

    # 5. Remove spaces around line breaks
    text = re.sub(r" *\n *", "\n", text)

    # 6. Reduce excessive consecutive line breaks
    text = re.sub(r"\n{3,}", "\n\n", text)

    # 7. Remove leading and trailing whitespace
    text = text.strip()

    return text

In [ ]:
jd_clean = clean_text(jd_text)
print(jd_clean)

In [ ]:
resume_clean = clean_text(resume_text)
print(resume_clean)

# Step 3 Key features extraction

## Skills extraction

In [ ]:
from ojd_daps_skills.extract_skills.extract_skills import SkillsExtractor

skills_extractor = SkillsExtractor(
    taxonomy_name="esco"
)

In [ ]:
# Input: Cleaned Job Description from Step 2
job_ads = [jd_clean]

# Extract skills
results = skills_extractor(job_ads)

# Get the first JD result
doc = results[0]

In [ ]:
print("===== NER Entities =====")

for ent in doc.ents:
    print(f"{ent.text:<45} {ent.label_}")

In [ ]:
# Extract skills
jd_skills = list(dict.fromkeys(doc._.skill_spans))

# Remove duplicates
jd_skills = list(dict.fromkeys(jd_skills))

print("===== Extracted Skills =====")

for i, skill in enumerate(jd_skills, start=1):
    print(f"{i}. {skill}")

## Experience & Education Extraction

In [ ]:
# Regex + spaCy
import re
import spacy
import pandas as pd

nlp_sent = spacy.blank("en")
nlp_sent.add_pipe("sentencizer")

### Experience extraction

In [ ]:
EXPERIENCE_PATTERNS = [
    # At least 3 years / Minimum 3 years
    (
        "minimum",
        re.compile(
            r"\b(?:at\s+least|minimum(?:\s+of)?|min\.?)\s*"
            r"(?P<min>\d+(?:\.\d+)?)\s*\+?\s*"
            r"(?:years?|yrs?)\b",
            re.IGNORECASE
        )
    ),

    # 3-5 years / 3 to 5 years
    (
        "range",
        re.compile(
            r"\b(?P<min>\d+(?:\.\d+)?)\s*"
            r"(?:-|–|—|to)\s*"
            r"(?P<max>\d+(?:\.\d+)?)\s*"
            r"(?:years?|yrs?)\b",
            re.IGNORECASE
        )
    ),

    # 3+ years / 5+ yrs
    (
        "plus",
        re.compile(
            r"\b(?P<min>\d+(?:\.\d+)?)\s*\+\s*"
            r"(?:years?|yrs?)\b",
            re.IGNORECASE
        )
    ),

    # Over 4 years / More than 4 years
    (
        "greater_than",
        re.compile(
            r"\b(?:over|more\s+than)\s*"
            r"(?P<min>\d+(?:\.\d+)?)\s*"
            r"(?:years?|yrs?)\b",
            re.IGNORECASE
        )
    ),

    # 3 years of experience
    (
        "exact",
        re.compile(
            r"\b(?P<min>\d+(?:\.\d+)?)\s*"
            r"(?:years?|yrs?)\s+"
            r"(?:of\s+)?(?:relevant\s+|professional\s+|"
            r"working\s+|hands-on\s+|practical\s+)*"
            r"(?:experience|exp\.?)\b",
            re.IGNORECASE
        )
    )
]

In [ ]:
def extract_experience(text):
    """
    Extract numeric experience requirements from a text segment.

    Returns:
        list[dict]
    """

    results = []
    occupied_spans = []

    for pattern_type, pattern in EXPERIENCE_PATTERNS:

        for match in pattern.finditer(text):

            start, end = match.span()

            # Avoid duplicate extraction from overlapping patterns
            if any(
                start < existing_end and end > existing_start
                for existing_start, existing_end in occupied_spans
            ):
                continue

            min_years = float(match.group("min"))

            max_value = match.groupdict().get("max")
            max_years = (
                float(max_value)
                if max_value is not None
                else None
            )

            results.append({
                "category": "Experience",
                "requirement": match.group(0),
                "min_years": min_years,
                "max_years": max_years,
                "comparison": pattern_type,
                #"evidence": text.strip()
            })

            occupied_spans.append((start, end))

    return sorted(
        results,
        key=lambda item: text.find(item["requirement"])
    )

### Education extraction

In [ ]:
import pandas as pd
import spacy
import re

courses_df = pd.read_csv("Courses.csv")

# Remove missing values and repeated entries
courses = (
    courses_df["Courses"]
    .dropna()
    .astype(str)
    .str.strip()
)

# Remove duplicate course names (case-insensitive)
courses = courses.groupby(courses.str.casefold(), sort=False).first()

courses = courses.tolist()

print("Total courses:", len(courses))
print(courses[:20])

In [ ]:
from spacy.matcher import PhraseMatcher

nlp_field = spacy.blank("en")

field_matcher = PhraseMatcher(
    nlp_field.vocab,
    attr="LOWER"
)

patterns = [
    nlp_field.make_doc(course)
    for course in courses
]

field_matcher.add("FIELD_OF_STUDY", patterns)

print("Field matcher initialized.")

In [ ]:
EXTRA_FIELDS = [
    "Machine Learning",
    "Computational Linguistics"
]

extra_patterns = [
    nlp_field.make_doc(field)
    for field in EXTRA_FIELDS
]

field_matcher.add(
    "FIELD_OF_STUDY",
    extra_patterns
)

In [ ]:
EDUCATION_PATTERNS = {
    "PhD": [
        r"\bPh\.?\s*D\.?\b",             # PhD, Ph.D., Ph D
        r"\bdoctoral\s+degree\b",       # Doctoral degree
        r"\bdoctorate\b",               # Doctorate
        r"\bdoctor\s+of\s+philosophy\b"  # Doctor of Philosophy
    ],

    "Master": [
        r"\bmaster['’]?s?\s+degree\b",   # Master's degree, Master’s degree, Master degree
        r"\bmaster['’]?s?\b",           # Master, Master's, Master’s
        r"\bM\.?\s*Sc\.?\b",             # MSc, M.Sc.
        r"\bM\.?\s*Eng\.?\b",            # MEng, M.Eng.
        r"\bM\.?\s*A\.?\b",              # MA, M.A.
        r"\bMBA\b"                      # MBA
    ],

    "Bachelor": [
        r"\bbachelor['’]?s?\s+degree\b", # Bachelor's degree, Bachelor’s degree
        r"\bbachelor['’]?s?\b",         # Bachelor, Bachelor's, Bachelor’s
        r"\bB\.?\s*Sc\.?\b",             # BSc, B.Sc.
        r"\bB\.?\s*Eng\.?\b",            # BEng, B.Eng.
        r"\bB\.?\s*A\.?\b",              # BA, B.A.
        r"\bB\.?\s*B\.?\s*A\.?\b",       # BBA, B.B.A.
        r"\bundergraduate\s+degree\b"   # Undergraduate degree
    ],

    "Diploma": [
        r"\bdiploma\b",                 # Diploma
        r"\badvanced\s+diploma\b",      # Advanced Diploma
        r"\bgraduate\s+diploma\b",      # Graduate Diploma
        r"\bpostgraduate\s+diploma\b",  # Postgraduate Diploma
        r"\bassociate['’]?s?\s+degree\b" # Associate's degree
    ]
}

EDUCATION_CONTEXT = re.compile(
    r"\b("
    r"degree|bachelor|master|phd|doctorate|diploma|"
    r"qualification|qualifications|"
    r"academic background|educational background|"
    r"field of study|related field|related discipline|"
    r"graduate(?:d)? in"
    r")\b",
    re.IGNORECASE
)

def extract_fields_of_study(text):
    """
    Extract fields of study using Courses.csv
    and spaCy PhraseMatcher.
    """

    doc = nlp_field(text)
    matches = field_matcher(doc)

    extracted = []

    for match_id, start, end in matches:

        span = doc[start:end]

        extracted.append({
            "field": span.text,
            "start": span.start_char,
            "end": span.end_char
        })

    # Prefer longer matches to avoid overlapping results
    extracted.sort(
        key=lambda x: (
            -(x["end"] - x["start"]),
            x["start"]
        )
    )

    selected = []

    for item in extracted:

        overlap = any(
            item["start"] < existing["end"]
            and item["end"] > existing["start"]
            for existing in selected
        )

        if not overlap:
            selected.append(item)

    return [
        item["field"]
        for item in sorted(selected, key=lambda x: x["start"])
    ]

In [ ]:
def extract_education(text):
    """
    Extract education requirements, including cases
    where the degree level is not specified.
    """

    if not EDUCATION_CONTEXT.search(text):
        return []

    # 1. Extract degree levels
    degree_matches = []

    for level, patterns in EDUCATION_PATTERNS.items():
        for pattern in patterns:
            for match in re.finditer(pattern, text, re.IGNORECASE):
                degree_matches.append({
                    "level": level,
                    "text": match.group(0),
                    "start": match.start(),
                    "end": match.end()
                })

    # Prefer longer matches
    degree_matches.sort(
        key=lambda x: (-(x["end"] - x["start"]), x["start"])
    )

    selected_degrees = []

    for degree in degree_matches:
        if not any(
            degree["start"] < d["end"] and
            degree["end"] > d["start"]
            for d in selected_degrees
        ):
            selected_degrees.append(degree)

    # Extract fields of study using Courses.csv
    fields = extract_fields_of_study(text)

    # 3. Return even if no degree level is specified
    if not selected_degrees:
        return [{
            "category": "Education",
            "requirement": "Unspecified qualification",
            "education_level": None,
            "field_of_study": fields,
            #"evidence": text.strip()
        }]

    results = []

    for degree in selected_degrees:
        results.append({
            "category": "Education",
            "requirement": degree["text"],
            "education_level": degree["level"],
            "field_of_study": fields,
            #"evidence": text.strip()
        })

    return results

### Experience and Education output

In [ ]:
# Split JD into segments
def split_jd_segments(text):
    """
    Split JD text by lines and sentence boundaries.
    """

    segments = []

    for line in text.splitlines():
        line = line.strip()

        if not line:
            continue

        # Remove common bullet prefixes
        line = re.sub(r"^[•●▪\-*]\s*", "", line)

        doc = nlp_sent(line)

        for sent in doc.sents:
            sentence = sent.text.strip()

            if sentence:
                segments.append(sentence)

    return segments

# Run Extraction 
def extract_jd_experience_education(jd_text):

    segments = split_jd_segments(jd_text)

    experience_results = []
    education_results = []

    for segment in segments:

        experience_results.extend(
            extract_experience(segment)
        )

        education_results.extend(
            extract_education(segment)
        )

    return experience_results, education_results

In [ ]:
jd_experience, jd_education = (
    extract_jd_experience_education(jd_clean)
)
experience_df = pd.DataFrame(jd_experience)
education_df = pd.DataFrame(jd_education)

print("===== EXPERIENCE REQUIREMENTS =====")
display(experience_df)

print("===== EDUCATION REQUIREMENTS =====")
display(education_df)

## Requirement Classification (Mandatory / Preferred)

In [ ]:
# Rule-Based Classification + Context Analysis

PREFERRED_PATTERNS = [
    r"\bpreferred\b",
    r"\bpreferably\b",
    r"\bwould be preferred\b",
    r"\bwould be advantageous\b",
    r"\badvantageous\b",
    r"\ban advantage\b",
    r"\ba plus\b",
    r"\bbonus\b",
    r"\bnice to have\b",
    r"\bgood to have\b",
    r"\bdesirable\b",
    r"\bnot mandatory\b",
    r"\bnot required\b",
    r"\boptional\b",
    r"\bnot essential\b",
    r"\bhighly regarded\b",
    r"\bwill be given preference\b",
]

REQUIRED_PATTERNS = [
    r"\brequired\b",
    r"\bmust\b",
    r"\bmandatory\b",
    r"\bessential\b",
    r"\bminimum\b",
    r"\bat least\b",
    r"\bneed(?:s)? to\b",
    r"\bnecessary\b",
    r"\bexpected to have\b",
    r"\bproficiency in\b",
    r"\bproficient in\b",
]

MANDATORY_PATTERNS = [
    r"\bwithout .*? (?:not eligible|ineligible)\b",
    r"\bnot eligible\b",
    r"\bineligible\b",
    r"\bwill not be considered\b",
    r"\bapplications? will be rejected\b",
    r"\bnon[- ]negotiable\b",
    r"\bstrict eligibility requirement\b",
]

In [ ]:
def contains_pattern(text, patterns):
    """Check whether text matches any regex pattern."""
    return any(
        re.search(pattern, text, flags=re.IGNORECASE)
        for pattern in patterns
    )


def classify_requirement(text):
    """
    Classify requirement priority based on explicit language.

    Returns:
        MANDATORY, REQUIRED, PREFERRED or UNSPECIFIED
    """

    text = text.strip()

    # Preferred language and explicit negated requirements
    if contains_pattern(text, PREFERRED_PATTERNS):
        return "PREFERRED"

    # Explicit eligibility restrictions
    if contains_pattern(text, MANDATORY_PATTERNS):
        return "MANDATORY"

    # Required language
    if contains_pattern(text, REQUIRED_PATTERNS):
        return "REQUIRED"

    return "UNSPECIFIED"

In [ ]:
def split_requirement_clauses(text):
    """
    Split a sentence into clauses using common
    contrast markers and semicolons.
    """

    clauses = re.split(
        r"\s*(?:;|\bwhile\b|\bwhereas\b)\s*",
        text,
        flags=re.IGNORECASE
    )

    return [
        clause.strip(" ,.;")
        for clause in clauses
        if clause.strip(" ,.;")
    ]

In [ ]:
def find_skill_contexts(jd_text, skills):
    """
    Find JD clauses containing extracted skills.

    Supports:
        - str
        - spaCy Span objects
    """

    segments = split_jd_segments(jd_text)

    results = []

    for segment in segments:

        clauses = split_requirement_clauses(segment)

        for clause in clauses:

            for skill in skills:

                # Convert skill to string
                if isinstance(skill, str):
                    skill_text = skill

                elif hasattr(skill, "text"):
                    skill_text = skill.text

                else:
                    continue

                skill_text = skill_text.strip()

                if not skill_text:
                    continue

                # Escape special characters
                pattern = (
                    r"(?<!\w)"
                    + re.escape(skill_text)
                    + r"(?!\w)"
                )

                if re.search(pattern, clause, re.IGNORECASE):

                    results.append({
                        "category": "Skill",
                        "requirement": skill_text,
                        "priority": classify_requirement(clause),
                        "evidence": clause
                    })

    return results

In [ ]:
def classify_extracted_requirements(requirements):
    """
    Add priority classification to extracted
    experience or education requirements.
    """

    results = []

    for item in requirements:

        result = item.copy()

        evidence = result.get("evidence", "")

        clauses = split_requirement_clauses(evidence)

        # Determine clauses that contain the extracted requirement
        requirement_text = str(result.get("requirement", ""))

        matching_clauses = [
            clause
            for clause in clauses
            if requirement_text.lower() in clause.lower()
        ]

        if len(matching_clauses) == 1:
            context = matching_clauses[0]
        else:
            context = evidence

        result["priority"] = classify_requirement(context)

        results.append(result)

    return results

In [ ]:
# 1. Classify Skills
classified_skills = find_skill_contexts(
    jd_clean,
    jd_skills
)

# 2. Classify Experience
classified_experience = classify_extracted_requirements(
    jd_experience
)

# 3. Classify Education
classified_education = classify_extracted_requirements(
    jd_education
)

# 4. Combine all requirements
all_requirements = (
    classified_skills
    + classified_experience
    + classified_education
)

# 5. Create DataFrame
jd_requirements_df = pd.DataFrame(all_requirements)

display(jd_requirements_df)

In [ ]:
jd_requirements_df.to_csv(
    "jd_requirements_classified.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Requirement classification completed.")

## Structured JD Requirements

In [ ]:
REQUIREMENT_COLUMNS = [
    "requirement_id",
    "category",
    "requirement",
    "priority",
    "min_years",
    "max_years",
    "education_level",
    "field_of_study",
    "evidence"
]
all_requirements = (
    classified_skills
    + classified_experience
    + classified_education
)

jd_requirements_df = pd.DataFrame(all_requirements)

# Add any missing columns
for col in REQUIREMENT_COLUMNS:
    if col not in jd_requirements_df.columns:
        jd_requirements_df[col] = None

jd_requirements_df["priority"] = (
    jd_requirements_df["priority"]
    .fillna("UNSPECIFIED")
    .astype(str)
    .str.upper()
)

jd_requirements_df = jd_requirements_df.drop_duplicates(
    subset=["category", "requirement", "priority", "evidence"]
).reset_index(drop=True)

In [ ]:
jd_requirements_df["requirement_id"] = [
    f"R{i:03d}"
    for i in range(1, len(jd_requirements_df) + 1)
]

jd_requirements_df = jd_requirements_df[
    REQUIREMENT_COLUMNS
]

display(jd_requirements_df)

In [ ]:
jd_requirements_df.to_csv(
    "jd_requirements_structured.csv",
    index=False,
    encoding="utf-8-sig"
)

# Step 4 Resume details extraction

In [ ]:
resume_docs = skills_extractor([resume_clean])
resume_doc = resume_docs[0]

resume_skills = []

for item in resume_doc._.skill_spans:
    if isinstance(item, str):
        skill = item
    elif hasattr(item, "text"):
        skill = item.text
    else:
        continue

    skill = skill.strip()

    if skill:
        resume_skills.append(skill)

resume_skills = list(dict.fromkeys(resume_skills))

print("===== RESUME SKILLS =====")
for skill in resume_skills:
    print("-", skill)

In [ ]:
resume_skills_df = pd.DataFrame(
    resume_doc._.mapped_skills
)

display(resume_skills_df)

In [ ]:
EDUCATION_PATTERNS["Master"].extend([
    r"\bmaster\s+of\s+[\w\s&-]+",
    r"\bMComp\b",
    r"\bMTech\b"
])

EDUCATION_PATTERNS["Bachelor"].extend([
    r"\bbachelor\s+of\s+[\w\s&-]+",
    r"\bBTech\b"
])

#这里 Master of ... 和 Bachelor of ... 的匹配范围较宽，因此建议保留原始证据，必要时检查结果。以后可以用经过人工标注的数据改善这一部分

In [ ]:
resume_education = []

for segment in split_jd_segments(resume_clean):

    results = extract_education(segment)

    resume_education.extend(results)

resume_education_df = pd.DataFrame(resume_education)

display(resume_education_df)

In [ ]:
SECTION_ALIASES = {
    "education": [
        "education",
        "academic background",
        "academic qualifications",
        "educational background"
    ],
    "experience": [
        "experience",
        "work experience",
        "professional experience",
        "employment history",
        "work history"
    ],
    "projects": [
        "projects",
        "project experience",
        "academic projects",
        "personal projects"
    ],
    "skills": [
        "skills",
        "technical skills",
        "core competencies"
    ],
    "summary": [
        "summary",
        "profile",
        "professional summary",
        "objective"
    ]
}


def split_resume_sections(text):
    sections = {}
    current_section = "other"

    sections[current_section] = []

    for line in text.splitlines():

        line = line.strip()

        if not line:
            continue

        normalized = line.lower().rstrip(":").strip()

        detected_section = None

        for section_name, aliases in SECTION_ALIASES.items():
            if normalized in aliases:
                detected_section = section_name
                break

        if detected_section:
            current_section = detected_section
            sections.setdefault(current_section, [])
        else:
            sections.setdefault(current_section, [])
            sections[current_section].append(line)

    return {
        name: "\n".join(lines)
        for name, lines in sections.items()
    }

In [ ]:
resume_sections = split_resume_sections(resume_clean)

for section, content in resume_sections.items():
    print(f"\n===== {section.upper()} =====")
    print(content[:800])

In [ ]:
# Resume Work Experience Extraction
MONTHS = (
    r"Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|"
    r"Apr(?:il)?|May|Jun(?:e)?|Jul(?:y)?|"
    r"Aug(?:ust)?|Sep(?:tember)?|Oct(?:ober)?|"
    r"Nov(?:ember)?|Dec(?:ember)?"
)

DATE_TOKEN = (
    rf"(?:{MONTHS})\.?\s+\d{{4}}"
    rf"|\d{{4}}[/-](?:0?[1-9]|1[0-2])"
)

DATE_RANGE_PATTERN = re.compile(
    rf"(?P<start>{DATE_TOKEN})"
    rf"\s*(?:-|–|—|\bto\b)\s*"
    rf"(?P<end>{DATE_TOKEN}|\bPresent\b|\bCurrent\b|\bNow\b)",
    re.IGNORECASE
)

In [ ]:
def extract_experience_entries(experience_text):
    """
    Extract dated experience entries.

    The surrounding text is retained as evidence.
    Job title/company extraction is not yet performed.
    """

    matches = list(DATE_RANGE_PATTERN.finditer(experience_text))

    entries = []

    for i, match in enumerate(matches):

        # The text between the previous date and this date
        # may contain the job title and company.
        previous_end = (
            matches[i - 1].end()
            if i > 0 else 0
        )

        next_start = (
            matches[i + 1].start()
            if i + 1 < len(matches)
            else len(experience_text)
        )

        heading_context = experience_text[
            previous_end:match.start()
        ].strip()

        description = experience_text[
            match.end():next_start
        ].strip()

        entries.append({
            "start_date": match.group("start"),
            "end_date": match.group("end"),
            "date_range": match.group(0),
            "heading_context": heading_context,
            "description": description
        })

    return entries

In [ ]:
experience_text = resume_sections.get("experience", "")

resume_experience = extract_experience_entries(
    experience_text
)

resume_experience_df = pd.DataFrame(resume_experience)

display(resume_experience_df)

In [ ]:
projects_text = resume_sections.get("projects", "")

print("===== PROJECT EXPERIENCE =====")
print(projects_text)

In [ ]:
resume_evidence = []

for section_name in ["summary", "experience", "projects", "skills"]:

    section_text = resume_sections.get(section_name, "")

    if not section_text.strip():
        continue

    for segment in split_jd_segments(section_text):

        resume_evidence.append({
            "source_section": section_name,
            "evidence": segment
        })

resume_evidence_df = pd.DataFrame(resume_evidence)

display(resume_evidence_df)

In [ ]:
# Create Resume Profile
resume_profile = {
    "skills": resume_skills,
    "skill_mappings": resume_skills_df.to_dict(
        orient="records"
    ),
    "education": resume_education,
    "experience": resume_experience,
    "evidence": resume_evidence
}

print("Resume profile created.")

# Step 5 JD–Resume Semantic Matching

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cpu"
)

print("Sentence Transformer loaded.")

In [ ]:
# Copy original data
jd_df = jd_requirements_df.copy()

# Ensure expected columns
required_columns = [
    "requirement_id",
    "category",
    "requirement",
    "priority",
    "evidence",
    "min_years",
    "education_level",
    "field_of_study"
]

for col in required_columns:
    if col not in jd_df.columns:
        jd_df[col] = None

jd_df["priority"] = (
    jd_df["priority"]
    .fillna("UNSPECIFIED")
    .astype(str)
    .str.upper()
)

# Normalize skills into strings
def to_skill_string(item):
    if isinstance(item, str):
        return item.strip()
    if hasattr(item, "text"):
        return item.text.strip()
    return ""

resume_skills = list(dict.fromkeys(
    skill
    for item in resume_profile.get("skills", [])
    if (skill := to_skill_string(item))
))

print("JD Requirements:", len(jd_df))
print("Resume Skills:", len(resume_skills))
print("Resume Evidence:", len(resume_profile.get("evidence", [])))

In [ ]:
# Resume Evidence
def collect_resume_evidence(resume_profile):

    evidence_records = []

    # Explicit skills
    for skill in resume_profile.get("skills", []):
        text = to_skill_string(skill)

        if text:
            evidence_records.append({
                "text": text,
                "source": "skills"
            })

    # Project / work / summary evidence
    for item in resume_profile.get("evidence", []):

        if not isinstance(item, dict):
            continue

        text = str(item.get("evidence", "")).strip()

        if text:
            evidence_records.append({
                "text": text,
                "source": item.get(
                    "source_section",
                    "unknown"
                )
            })

    # Remove duplicates
    unique = {}
    for item in evidence_records:
        key = (item["text"].casefold(), item["source"])
        unique[key] = item

    return list(unique.values())


resume_evidence = collect_resume_evidence(resume_profile)

print("Total evidence records:", len(resume_evidence))

display(pd.DataFrame(resume_evidence).head(10))

In [ ]:
# Semantic Skill Matching
def exact_skill_match(skill, text):
    """
    Check if a skill is explicitly mentioned.
    """

    if not skill or not text:
        return False

    pattern = (
        r"(?<!\w)"
        + re.escape(skill)
        + r"(?!\w)"
    )

    return bool(
        re.search(pattern, text, re.IGNORECASE)
    )

In [ ]:
# Semantic Similarity
evidence_texts = [
    item["text"]
    for item in resume_evidence
]

if evidence_texts:
    evidence_embeddings = model.encode(
        evidence_texts,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True
    )
else:
    evidence_embeddings = np.empty((0, 384))

In [ ]:
def retrieve_evidence(
    requirement,
    evidence_records,
    evidence_embeddings,
    top_k=5
):
    """
    Retrieve most semantically relevant resume evidence.
    """

    if not evidence_records:
        return []

    query_embedding = model.encode(
        [requirement],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    similarities = cosine_similarity(
        query_embedding,
        evidence_embeddings
    )[0]

    ranked_indices = np.argsort(
        similarities
    )[::-1][:top_k]

    return [
        {
            "evidence": evidence_records[i]["text"],
            "source": evidence_records[i]["source"],
            "semantic_similarity": float(similarities[i])
        }
        for i in ranked_indices
    ]

In [ ]:
NEGATIVE_EVIDENCE_PATTERNS = [
    r"\bno experience (?:with|in|using)\b",
    r"\bwithout experience (?:with|in|using)\b",
    r"\bnot familiar with\b",
    r"\bnever used\b"
]


def is_negative_evidence(text):
    """
    Conservative detection of obvious negative evidence.
    """
    return any(
        re.search(pattern, text, re.IGNORECASE)
        for pattern in NEGATIVE_EVIDENCE_PATTERNS
    )


def semantic_to_score(similarity):
    """
    Experimental baseline mapping.
    Not a calibrated probability.
    """

    similarity = max(0.0, min(1.0, similarity))

    if similarity >= 0.80:
        return 85.0

    elif similarity >= 0.65:
        return 70.0

    elif similarity >= 0.50:
        return 50.0

    elif similarity >= 0.35:
        return 25.0

    return 0.0

In [ ]:
def match_skill(
    skill,
    resume_evidence,
    evidence_embeddings
):

    candidates = retrieve_evidence(
        skill,
        resume_evidence,
        evidence_embeddings,
        top_k=10
    )

    if not candidates:
        return {
            "score": 0.0,
            "match_type": "NOT_EVIDENCED",
            "evidence": None,
            "similarity": None
        }

    # Ignore obviously negative evidence in this baseline
    usable = [
        c for c in candidates
        if not is_negative_evidence(c["evidence"])
    ]

    if not usable:
        return {
            "score": 0.0,
            "match_type": "NEEDS_REVIEW",
            "evidence": None,
            "similarity": None
        }

    # Direct skill evidence
    exact_candidates = [
        c for c in usable
        if exact_skill_match(skill, c["evidence"])
    ]

    if exact_candidates:

        # Prefer explicit skills or project/work evidence
        best = max(
            exact_candidates,
            key=lambda c: (
                c["source"] in [
                    "experience", "projects", "skills"
                ],
                c["semantic_similarity"]
            )
        )

        return {
            "score": 100.0,
            "match_type": "EXACT",
            "evidence": best["evidence"],
            "similarity": best["semantic_similarity"]
        }

    # Semantic evidence
    best = usable[0]

    score = semantic_to_score(
        best["semantic_similarity"]
    )

    match_type = (
        "RELATED"
        if score >= 70
        else "PARTIAL"
        if score >= 40
        else "WEAK"
        if score > 0
        else "NOT_EVIDENCED"
    )

    return {
        "score": score,
        "match_type": match_type,
        "evidence": best["evidence"],
        "similarity": best["semantic_similarity"]
    }

In [ ]:
# education scoring matrix
EDUCATION_RANK = {
    "Diploma": 1,
    "Bachelor": 2,
    "Master": 3,
    "PhD": 4
}


def normalize_degree_level(level):
    if level is None:
        return None

    level = str(level).strip().lower()

    if level in ["", "none", "nan"]:
        return None

    if "phd" in level or "doctor" in level:
        return "PhD"

    if "master" in level:
        return "Master"

    if "bachelor" in level or "undergraduate" in level:
        return "Bachelor"

    if "diploma" in level or "associate" in level:
        return "Diploma"

    return None

In [ ]:
def safe_list(value):
    """
    Convert a field-of-study value into a list.
    """
    if value is None:
        return []

    if isinstance(value, (list, tuple, set)):
        return [str(x).strip() for x in value if str(x).strip()]

    if isinstance(value, str):
        if not value.strip():
            return []

        # Handle JSON-encoded lists
        if value.strip().startswith("["):
            try:
                parsed = json.loads(value)
                if isinstance(parsed, list):
                    return [str(x).strip() for x in parsed]
            except json.JSONDecodeError:
                pass

        return [value.strip()]

    return []


def match_education(jd_requirement, resume_education):

    required_level = normalize_degree_level(
        jd_requirement.get("education_level")
    )

    required_fields = safe_list(
        jd_requirement.get("field_of_study")
    )

    # No extracted education
    if not resume_education:
        return {
            "score": None,
            "match_type": "UNKNOWN",
            "evidence": None
        }

    best_score = 0.0
    best_evidence = None
    verified = False

    for item in resume_education:

        candidate_level = normalize_degree_level(
            item.get("education_level")
        )

        candidate_fields = safe_list(
            item.get("field_of_study")
        )

        # Degree level
        if required_level is None:
            level_score = 100.0

        elif candidate_level is None:
            continue

        elif (
            EDUCATION_RANK[candidate_level]
            >= EDUCATION_RANK[required_level]
        ):
            level_score = 100.0

        else:
            level_score = 0.0

        # Field of study
        if not required_fields:
            field_score = 100.0

        elif not candidate_fields:
            continue

        else:
            # OR logic: one matching field is enough
            exact_field = any(
                jd_field.casefold() == resume_field.casefold()
                for jd_field in required_fields
                for resume_field in candidate_fields
            )

            field_score = 100.0 if exact_field else 0.0

        score = min(level_score, field_score)
        verified = True

        if best_evidence is None or score > best_score:
            best_score = score
            best_evidence = item.get("evidence")

    if not verified:
        return {
            "score": None,
            "match_type": "UNKNOWN",
            "evidence": None
        }

    return {
        "score": best_score,
        "match_type": (
            "MATCHED" if best_score == 100
            else "NOT_MATCHED"
        ),
        "evidence": best_evidence
    }

In [ ]:
from datetime import datetime, date
import calendar


def parse_resume_date(text):
    if not text:
        return None

    text = str(text).strip()

    if text.lower() in ["present", "current", "now"]:
        return date.today().replace(day=1)

    formats = [
        "%B %Y",
        "%b %Y",
        "%b. %Y",
        "%Y-%m",
        "%Y/%m"
    ]

    for fmt in formats:
        try:
            return datetime.strptime(text, fmt).date()
        except ValueError:
            continue

    return None

In [ ]:
def month_index(d):
    return d.year * 12 + d.month


def calculate_experience_years(experience_entries):

    intervals = []
    unparsed_count = 0

    for item in experience_entries:

        start = parse_resume_date(
            item.get("start_date")
        )

        end = parse_resume_date(
            item.get("end_date")
        )

        if start is None or end is None:
            unparsed_count += 1
            continue

        start_month = month_index(start)
        end_month = month_index(end)

        if end_month < start_month:
            unparsed_count += 1
            continue

        # Inclusive month interval
        intervals.append((start_month, end_month))

    if not intervals:
        return {
            "years": None,
            "unparsed_entries": unparsed_count
        }

    intervals.sort()

    merged = []

    for start, end in intervals:

        if not merged or start > merged[-1][1] + 1:
            merged.append([start, end])
        else:
            merged[-1][1] = max(
                merged[-1][1],
                end
            )

    total_months = sum(
        end - start + 1
        for start, end in merged
    )

    return {
        "years": round(total_months / 12, 2),
        "unparsed_entries": unparsed_count
    }

In [ ]:
def match_experience(jd_requirement, experience_summary):

    required_years = jd_requirement.get("min_years")

    if required_years is None or pd.isna(required_years):
        return {
            "score": None,
            "match_type": "UNKNOWN",
            "evidence": "No numeric requirement was extracted."
        }

    actual_years = experience_summary.get("years")

    if actual_years is None:
        return {
            "score": None,
            "match_type": "UNKNOWN",
            "evidence": "No parseable employment dates."
        }

    required_years = float(required_years)

    # This is a preliminary duration check, not
    # verification of relevant professional experience.
    if actual_years >= required_years:
        return {
            "score": None,
            "match_type": "DURATION_MET_RELEVANCE_UNVERIFIED",
            "evidence": (
                f"{actual_years} years of dated experience; "
                f"requires {required_years} years."
            )
        }

    return {
        "score": None,
        "match_type": "DURATION_BELOW_REQUIREMENT_UNVERIFIED",
        "evidence": (
            f"{actual_years} years of dated experience; "
            f"requires {required_years} years. "
            f"Work history may be incomplete."
        )
    }

In [ ]:
# Requirement Matching Pipeline
def match_jd_requirements(
    jd_df,
    resume_profile,
    resume_evidence,
    evidence_embeddings
):

    experience_summary = calculate_experience_years(
        resume_profile.get("experience", [])
    )

    results = []

    for _, row in jd_df.iterrows():

        req = row.to_dict()

        category = str(req.get("category", "")).lower()
        requirement = str(req.get("requirement", "")).strip()

        if category == "skill":

            result = match_skill(
                requirement,
                resume_evidence,
                evidence_embeddings
            )

        elif category == "education":

            result = match_education(
                req,
                resume_profile.get("education", [])
            )

        elif category == "experience":

            result = match_experience(
                req,
                experience_summary
            )

        else:

            result = {
                "score": None,
                "match_type": "UNSUPPORTED",
                "evidence": None,
                "similarity": None
            }

        results.append({
            "requirement_id": req.get("requirement_id"),
            "category": req.get("category"),
            "requirement": requirement,
            "priority": req.get("priority"),
            "match_score": result.get("score"),
            "match_type": result.get("match_type"),
            "resume_evidence": result.get("evidence"),
            "semantic_similarity": result.get("similarity")
        })

    return pd.DataFrame(results)

In [ ]:
jd_resume_match_df = match_jd_requirements(
    jd_df=jd_df,
    resume_profile=resume_profile,
    resume_evidence=resume_evidence,
    evidence_embeddings=evidence_embeddings
)

display(jd_resume_match_df)

In [ ]:
# Mandatory Eligibility Check
def check_mandatory_requirements(match_df):

    mandatory_df = match_df[
        match_df["priority"] == "MANDATORY"
    ].copy()

    if mandatory_df.empty:
        return {
            "eligibility": "NO_MANDATORY_CONSTRAINTS",
            "details": mandatory_df
        }

    statuses = []

    for _, row in mandatory_df.iterrows():

        # Current extraction is not enough to verify
        # hard constraints automatically.
        # Leave unresolved cases for verification.
        status = "UNKNOWN"

        statuses.append(status)

    mandatory_df["eligibility_status"] = statuses

    if "FAIL" in statuses:
        overall = "NOT_QUALIFIED"

    elif "UNKNOWN" in statuses:
        overall = "PENDING_VERIFICATION"

    else:
        overall = "ELIGIBLE"

    return {
        "eligibility": overall,
        "details": mandatory_df
    }

In [ ]:
eligibility_result = check_mandatory_requirements(
    jd_resume_match_df
)

print("Eligibility:", eligibility_result["eligibility"])

display(eligibility_result["details"])

In [ ]:
# Compute weight
def calculate_weighted_score(match_df):

    base_df = match_df[
        match_df["priority"].isin(
            ["REQUIRED", "UNSPECIFIED"]
        )
    ].copy()

    if base_df.empty:
        return None

    # Do not silently ignore unknown scores
    if base_df["match_score"].isna().any():
        return None

    base_df["weight"] = (
        base_df["priority"]
        .map(PRIORITY_WEIGHTS)
    )

    base_df["normalized_weight"] = (
        base_df["weight"] /
        base_df["weight"].sum()
    )

    base_score = (
        base_df["match_score"]
        * base_df["normalized_weight"]
    ).sum()

    return round(float(base_score), 2)

In [ ]:
PRIORITY_WEIGHTS = {
    "MANDATORY": 1.0,
    "REQUIRED": 0.9,
    "UNSPECIFIED": 0.5,
    "PREFERRED": 0.3
}

In [ ]:
base_score = calculate_weighted_score(
    jd_resume_match_df
)

print("Base Match Score:", base_score)

In [ ]:
# Bonus condition
BONUS_MAX = 10.0

def calculate_bonus(match_df):

    preferred_df = match_df[
        match_df["priority"] == "PREFERRED"
    ].copy()

    if preferred_df.empty:
        return 0.0

    if preferred_df["match_score"].isna().any():
        return None

    avg_preferred_match = (
        preferred_df["match_score"].mean()
    )

    bonus = BONUS_MAX * avg_preferred_match / 100

    return round(float(bonus), 2)

In [ ]:
bonus_score = calculate_bonus(
    jd_resume_match_df
)

print("Bonus Score:", bonus_score)

In [ ]:
# Final score
def calculate_final_result(
    match_df,
    eligibility_result
):

    base = calculate_weighted_score(match_df)
    bonus = calculate_bonus(match_df)

    eligibility = eligibility_result["eligibility"]

    if eligibility == "NOT_QUALIFIED":

        return {
            "eligibility": eligibility,
            "base_score": base,
            "bonus_score": bonus,
            "final_score": 0.0
        }

    if base is None or bonus is None:

        return {
            "eligibility": eligibility,
            "base_score": base,
            "bonus_score": bonus,
            "final_score": None
        }

    final_score = min(
        100.0,
        base + bonus
    )

    return {
        "eligibility": eligibility,
        "base_score": base,
        "bonus_score": bonus,
        "final_score": round(final_score, 2),
        "provisional": (
            eligibility == "PENDING_VERIFICATION"
        )
    }

In [ ]:
final_result = calculate_final_result(
    jd_resume_match_df,
    eligibility_result
)

print("===== RESUME MATCHING RESULT =====")

for key, value in final_result.items():
    print(f"{key}: {value}")

In [ ]:
# save to csv
# Save requirement-level matching results
import json

jd_resume_match_df.to_csv(
    "Final_results.csv",
    index=False,
    encoding="utf-8-sig"
)

# Save overall matching result
with open(
    "matching_summary.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        final_result,
        f,
        ensure_ascii=False,
        indent=4
    )

print("Results saved successfully.")

# Part II — Evaluated system

## 0. Setup
Install once: `pip install -r requirements.txt` (and for the ESCO baseline `python -m spacy download en_core_web_sm`).

* `USE_DEMO = True` runs everything on the tiny **synthetic** demo set in `data/demo/` to check the code.
  **Demo numbers must not be reported.** Switch to `False` after Member A has collected and annotated real data.
* `RUN_HEAVY = True` enables transformer models (zero-shot NLI, DistilBERT, cross-encoder, fine-tuning) — use a GPU (a T4-class GPU is enough).

In [ ]:
import sys, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd()))

from src.pipeline import MatchingPipeline, PipelineConfig, gap_report, clean_text
from src import data_utils as du, skill_extraction as se, requirement_clf as rc
from src import matching as mt, scoring as sc, fairness as fa, plots
from src.nlp_backend import Embedder, HAS_SPACY, HAS_ST

USE_DEMO  = True
RUN_HEAVY = False
EMBEDDER  = "sentence-transformers/all-MiniLM-L6-v2" if HAS_ST else "tfidf-char"
DATA      = Path("data/demo") if USE_DEMO else Path("data")
Path("results").mkdir(exist_ok=True)
print("spaCy:", HAS_SPACY, "| sentence-transformers:", HAS_ST, "| embedder:", EMBEDDER)

## 1. Bug fixes found while building the evaluation (Part I → Part II)
| Problem observed in Part I | Fix |
|---|---|
| `NumPy` split into "Num" / "Py," by the PDF line break | `fix_broken_words` re-joins tokens whose concatenation is a known skill |
| Projects section contained phone/e-mail/LinkedIn | contact lines are routed to a separate `contact` section |
| Generic phrases extracted as skills ("national initiative driven", "workshops") | dictionary extractor + post-processing filter (Member B) |
| "NITEC, Diploma, Bachelor's, Master's or PhD" → four separate requirements (PhD effectively required) | alternatives joined by *or* → one requirement at the **lowest** level |
| "Nanyang Technological University" in the company intro → education requirement | education rows without degree/field context are dropped |
| Bullets under "Nice to have:" were UNSPECIFIED | sentences inherit the priority of their section header |
| `safe_list` used `json` before import; experience score always `None` | fixed; experience gets a proportional score |
| ALL-CAPS resume: glued layout not restored (no lower→Upper cues), upper-case name lines kept | `restore_caps_layout` + upper-case name filter, only for mostly upper-case resumes (`caps_fix`); normal-case resumes are scored exactly as before |

In [ ]:
print(clean_text("Programming: Python (Pandas, Num\nPy, Scikit-learn)"))
pipe = MatchingPipeline(PipelineConfig(embedder_name=EMBEDDER, skill_extractor="dictionary+esco"))
aisg = Path("data/demo/JD_example1_ntu_AI.txt").read_text(encoding="utf-8")
aisg_req = pipe.extract_jd_requirements(aisg)
display(aisg_req[["category", "requirement", "priority", "education_level"]])

---
## 2. Member A — Dataset construction & annotation

**Target size** (realistic for 6 people in ~2 weeks):
* 30–50 JDs from different roles (data / software / ML / business) — MyCareersFuture, LinkedIn, company sites
* 40–80 resumes — anonymised team/volunteer resumes + Kaggle *Resume Dataset* (`Resume.csv`)
* ~300 JD–resume pairs labelled **0 = No fit / 1 = Potential / 2 = Good fit** (each pair by 2 annotators)
* all JD sentences labelled MANDATORY / REQUIRED / PREFERRED / UNSPECIFIED (≥20% double-annotated)
* gold skill lists for ≥20 JDs and ≥20 resumes
* External: HuggingFace `cnamuangtoun/resume-job-description-fit` (labelled pairs) for training / extra testing.

Annotation guidelines: `data/annotation_guidelines.md`.

In [ ]:
jds     = du.load_folder(DATA / "jds")
resumes = du.load_folder(DATA / "resumes")
# Optional: add Kaggle resumes
# kaggle = du.load_kaggle_resumes("data/raw/Resume.csv", categories=["INFORMATION-TECHNOLOGY", "ENGINEERING"], n_per_category=20)
# resumes.update(dict(zip(kaggle.resume_id, kaggle.resume_text)))

In [ ]:
# Generate annotation sheets (run once, then annotate in Excel / Google Sheets)
if not USE_DEMO:
    du.make_requirement_sheet(jds, "data/annotations/requirements_to_label.csv")
    du.make_skill_sheet(jds, "data/annotations/jd_skills_to_label.csv", "JD", suggestions=se.DictionarySkillExtractor())
    du.make_skill_sheet(resumes, "data/annotations/resume_skills_to_label.csv", "Resume", suggestions=se.DictionarySkillExtractor())
    du.make_match_sheet(jds, resumes, "data/annotations/pairs_to_label.csv")

In [ ]:
# Load labels
match_file = DATA / ("demo_match_labels.csv" if USE_DEMO else "annotations/pairs_labelled.csv")
req_file   = DATA / ("demo_requirement_labels.csv" if USE_DEMO else "annotations/requirements_labelled.csv")
skill_file = DATA / ("demo_skill_gold.csv" if USE_DEMO else "annotations/jd_skills_labelled.csv")

labels = pd.read_csv(match_file)
if "annotator" in labels and labels.annotator.nunique() > 1:          # inter-annotator agreement
    display(du.agreement_report(labels.assign(item=labels.jd_id + "|" + labels.resume_id), "item", "label"))
    labels = du.adjudicate(labels.assign(item=labels.jd_id + "|" + labels.resume_id), "item", "label")
    labels[["jd_id", "resume_id"]] = labels["item"].str.split("|", expand=True)
pairs = labels.assign(jd_text=labels.jd_id.map(jds), resume_text=labels.resume_id.map(resumes)).dropna(subset=["jd_text", "resume_text"])
pairs["label"] = pairs["label"].astype(int)

req_labels = pd.read_csv(req_file)
if "annotator" in req_labels and req_labels.annotator.nunique() > 1:
    display(du.agreement_report(req_labels, "sent_id", "gold_label"))
    req_labels = du.adjudicate(req_labels, "sent_id", "gold_label").merge(
        req_labels.drop_duplicates("sent_id")[["sent_id", "jd_id", "sentence", "section_header"]], on="sent_id")

skill_gold = du.load_skill_gold(skill_file)
stats = du.dataset_statistics(jds, resumes, pairs); display(stats); stats.to_csv("results/A_dataset_stats.csv", index=False)

In [ ]:
# Optional: large external labelled set (needs internet + `datasets`)
if RUN_HEAVY:
    hf_pairs = du.load_hf_fit_pairs("train", n=1500)
    hf_test  = du.load_hf_fit_pairs("test",  n=600)
    print(hf_pairs.label.value_counts())

---
## 3. Member B — Skill extraction

| ID | Method |
|---|---|
| M1 | ESCO taxonomy extractor (`ojd_daps_skills`) — **Part I baseline** |
| M2 | M1 + post-processing (split enumerations, remove generic verb phrases, alias normalisation) |
| M3 | Curated dictionary (≈300 skills + aliases) with spaCy PhraseMatcher |
| M4 | M2 ∪ M3 |
| M5 | LLM few-shot extraction (JSON output) |
| M6 | JobBERT fine-tuned on SkillSpan (token classification) |

In [ ]:
extractors = {"M3_dictionary": se.DictionarySkillExtractor()}
try:
    extractors["M1_esco"] = se.ESCOSkillExtractor()
    extractors["M2_esco_filtered"] = se.ESCOFilteredExtractor()
    extractors["M4_combined"] = se.CombinedExtractor(extractors["M3_dictionary"], extractors["M2_esco_filtered"])
except ImportError:
    print("ojd_daps_skills not installed -> M1/M2/M4 skipped")

# M5 LLM (set an API key in the environment first)
# from src.llm_clients import openai_llm, anthropic_llm, hf_local_llm
# extractors["M5_llm"] = se.LLMSkillExtractor(hf_local_llm())

# M6 fine-tuned NER (GPU, ~15 min):
# se.train_skill_ner("models/skill_ner")
# extractors["M6_jobbert_skillspan"] = se.TokenClassifierExtractor("models/skill_ner")

docs = {d: jds.get(d) or resumes.get(d) for d in skill_gold if (jds.get(d) or resumes.get(d))}
skill_summary, skill_per_doc = se.evaluate_extractors(extractors, docs, skill_gold)
display(skill_summary); skill_summary.to_csv("results/B_skill_extraction.csv", index=False)
plots.grouped_bars(skill_summary, ["precision_soft", "recall_soft", "f1_soft"], title="Skill extraction (soft match)");

In [ ]:
# Error analysis for the best method: false positives / false negatives
best = skill_summary.sort_values("f1_soft").method.iloc[-1]
errs = se.extraction_errors(extractors[best], docs, skill_gold)
display(errs.groupby("type").skill.apply(lambda s: ", ".join(s.head(15))))

---
## 4. Member C — Requirement priority classification
Labels: MANDATORY merged into REQUIRED for evaluation (too few MANDATORY examples) → 3 classes.
**Label source:** the 749 gold labels in `data/annotations/requirements_labelled.csv` were produced by an LLM (Claude) following `data/annotation_guidelines.md` (column `annotator = claude`, single annotator, not human-verified), so the scores below measure agreement with LLM labels.
Trainable models use **GroupKFold by JD** so that sentences from one JD never appear in train and test.

In [ ]:
classifiers = {
    "C1_rules (Part I)":   lambda: rc.RuleClassifier(use_section_headers=False),
    "C2_rules+sections":   lambda: rc.RuleClassifier(use_section_headers=True),
    "C3_tfidf_lr":         lambda: rc.TfidfLRClassifier(),
}
if RUN_HEAVY:
    _zs = rc.ZeroShotNLIClassifier(labels=rc.LABELS_3, device=0)
    classifiers["C4_zero_shot_nli"] = lambda: _zs
    classifiers["C5_distilbert"] = lambda: rc.BertClassifier(epochs=6)

clf_summary, clf_preds = rc.evaluate_classifiers(classifiers, req_labels, n_splits=5)
display(clf_summary); clf_summary.to_csv("results/C_requirement_clf.csv", index=False)
plots.method_bar(clf_summary, "macro_f1", title="Requirement classification — macro-F1");

In [ ]:
fig, axes = plt.subplots(1, len(classifiers), figsize=(4.5 * len(classifiers), 4))
for ax, m in zip(np.atleast_1d(axes), classifiers):
    rc.plot_confusion(clf_preds, m, ax=ax)
plt.tight_layout()
display(rc.TfidfLRClassifier().fit(req_labels.sentence.tolist(), req_labels.gold_label.str.replace("MANDATORY", "REQUIRED").tolist(),
                                   req_labels.section_header.tolist()).top_features(8))

---
## 5. Member D — Matching models

In [ ]:
matchers = [mt.TfidfMatcher(), mt.BM25Matcher(),
            mt.EmbeddingMatcher(EMBEDDER, "doc"), mt.EmbeddingMatcher(EMBEDDER, "maxsim"),
            mt.PipelineMatcher(pipe, "D6_our_pipeline")]
if RUN_HEAVY:
    for name in ["sentence-transformers/all-mpnet-base-v2", "BAAI/bge-small-en-v1.5", "intfloat/e5-base-v2"]:
        matchers.append(mt.EmbeddingMatcher(name, "maxsim"))
    matchers.append(mt.CrossEncoderMatcher())

match_summary, match_scores = mt.evaluate_matchers(matchers, pairs, k=5)
display(match_summary); match_summary.to_csv("results/D_matching.csv", index=False)
match_scores.to_csv("results/D_pair_scores.csv", index=False)
plots.grouped_bars(match_summary, ["spearman", "macro_f1_cv", "ndcg@5"], title="JD–resume matching");

In [ ]:
# D7: fine-tune a bi-encoder on the external labelled pairs, then test on OUR annotated pairs (domain transfer)
if RUN_HEAVY:
    path = mt.fine_tune_biencoder(hf_pairs, epochs=1)
    ft = mt.EmbeddingMatcher(path, "doc"); ft.name = "D7_finetuned_biencoder"
    s_ours, _ = mt.evaluate_matchers([ft, mt.EmbeddingMatcher(EMBEDDER, "doc")], pairs)
    s_hf, _   = mt.evaluate_matchers([ft, mt.EmbeddingMatcher(EMBEDDER, "doc")], hf_test)
    display(s_ours.assign(test_set="ours"), s_hf.assign(test_set="hf_test"))

---
## 6. Member E — Score calibration, learned scorer, ablation

In [ ]:
results = sc.run_pipeline_on_pairs(pipe, pairs)
feat = sc.build_feature_table(results)
learned, coefs = sc.evaluate_learned_scorer(feat, n_splits=5)
display(learned); learned.to_csv("results/E_learned_scorer.csv", index=False)
display(coefs)   # which features drive "Good fit"?

In [ ]:
train_pairs, test_pairs = du.split_by_jd(pairs, test_size=0.3)
dev_mask = pairs.jd_id.isin(train_pairs.jd_id).values
grid_summary, grid_log = sc.grid_search_scoring(results, dev_mask)
display(grid_summary); display(grid_log.head(10))

In [ ]:
ablation = sc.run_ablation(pairs, PipelineConfig(embedder_name=EMBEDDER, skill_extractor="dictionary+esco"),
                           embedder=pipe.embedder, skill_extractor=pipe.skill_extractor)
display(ablation); ablation.to_csv("results/E_ablation.csv", index=False)
plots.method_bar(ablation, "spearman", label_col="variant", title="Ablation — Spearman ρ with human labels");

---
## 7. Member F — Fairness, robustness, error analysis, explanations

In [ ]:
some_jd = list(jds)[0]
sample  = dict(list(resumes.items())[:5])
name_summary, name_detail = fa.counterfactual_name_test(pipe, jds[some_jd], sample)
display(name_summary); name_summary.to_csv("results/F_name_fairness.csv", index=False)
display(fa.gender_wording_test(pipe, jds[some_jd], sample))

In [ ]:
rob_summary, rob_detail = fa.robustness_test(pipe, jds[some_jd], sample)
display(rob_summary); rob_summary.to_csv("results/F_robustness.csv", index=False)

In [ ]:
worst = fa.worst_pairs(match_scores, "D6_our_pipeline", k=5)
display(worst)
fa.error_annotation_sheet(worst, "results/F_errors_to_categorise.csv")   # categorise manually, see fa.ERROR_CATEGORIES
display(fa.misclassified_sentences(clf_preds, "C2_rules+sections").head(15))

In [ ]:
# Explanation for one candidate + ranking of all resumes for one JD
r = pipe.match(jds[some_jd], list(resumes.values())[0])
print(gap_report(r))
display(pipe.rank_resumes(jds[some_jd], resumes))
# from src.llm_clients import hf_local_llm; print(fa.llm_feedback(r, hf_local_llm()))

Demo UI: run `python app.py` in a terminal → upload a JD and a resume → score, requirement table, gap report.

---
## 8. Summary of results
Final numbers only; file index and provenance in `results/README.md`.

| Module | Benchmark | Baseline (Part I) | Best method | Gain |
|---|---|---|---|---|
| B Skill extraction | SkillSpan test, soft F1 | M4 dictionary+ESCO (Part I default) = 0.229; M1 ESCO = 0.131 | M6 JobBERT = 0.511 | +0.282 |
| C Requirement classification | 749 sentences (LLM-annotated), GroupKFold by JD, macro-F1 | C1 rules = 0.474 | C3b TF-IDF+LR = 0.651 (C5 DistilBERT = 0.634) | +0.177 |
| D Matching | external fit set, 600 pairs / 67 JDs, Spearman ρ | D6 our pipeline = 0.005 | D7 fine-tuned MiniLM = 0.281 (best untrained: BM25 = 0.143, 95% CI [0.000, 0.286]) | +0.276 |
| D Matching | TalentCLEF 2026 Task A dev, 10 queries, MAP | D6 pipeline v3 = 0.795 | mpnet doc similarity = 0.869; TF-IDF = 0.857; our hybrid D8 = 0.837 (gap to mpnet / TF-IDF not significant: p = 0.075 / 0.13) | hybrid vs D6: +0.042, 95% CI [0.018, 0.073] |
| D Matching | human benchmark (Vanetik & Kogan 2023), consensus Spearman ρ | D6 = −0.268 | BM25 = 0.283 (human–human agreement only 0.171) | — |
| E Scoring | fit set, Spearman ρ | E1 hand-set weights = −0.002 | E3a logistic regression = 0.128 | +0.130 |
| E Ablation | TalentCLEF dev, MAP (MiniLM), paired bootstrap | full hybrid = 0.837 | removing JD skill-list parsing → 0.589 (largest effect; exact / semantic match: not significant) | −0.248, 95% CI [0.093, 0.436] |
| F Fairness | counterfactual names, max abs Δ score | before fix = 8.13 | after fix = 0 (30 JDs, 80 cases); gender wording Δ = 0 | −8.13 |
| F Robustness | upper-casing the resume (fit set, 40 cases), mean / max abs Δ score | before fix: D6 = 6.72 / 50.0, D8 = 4.85 / 33.5 | after `caps_fix`: D6 = 3.02 / 37.5, D8 = 2.24 / 25.3 | about halved, not solved |

**Key findings.**
* Every component improves clearly over Part I (skills, requirement priorities, JD skill-list parsing).
* End-to-end ranking is harder: on TalentCLEF the requirement-level pipeline alone (0.795) is significantly worse than adding 0.5 × document similarity (0.837);
  the hybrid is still below mpnet (0.869) and TF-IDF (0.857), but with 10 queries these gaps are not significant (we cannot tell them apart).
  The pipeline's value is the explanation (requirement table, gap report), not raw ranking accuracy.
* On the human benchmark a predictor that ignores the CV entirely reaches ρ = 0.573 (`H_vacancy_prior_diagnostic.csv`):
  its labels mostly reflect the vacancy, not person–job fit, so it is a weak test of matching.
* The name-bias fix works by excluding the header/contact lines from scoring (blind screening), not by debiasing a model.

**Limitations:** TalentCLEF dev has only 10 queries (low test power; weight 2.0 > 0.5 is significant but was chosen on the same queries, so it is a trend, not a tuned value);
requirement-priority labels are LLM-annotated (single annotator); low human agreement on the fit labels; upper-casing still changes scores (about half as much after the fix); resumes mostly from one region/field;
LLM methods are not reproducible across API versions.

---
## 9. Independent extension — Evidence-based learning resource recommendation

The team pipeline answers **how the resume matches the JD**. This extension consumes its
requirement-level results and asks **which sourced learning units cover the skill evidence gaps**.
It does not modify the team's extraction, matching or scoring algorithms.

Research question: does coverage-aware selection reduce redundant topic assignments compared
with BM25, TF-IDF and lexical RRF ranking, and what prerequisite trade-offs arise?

**Run this section directly:** cells below are self-contained and do not require running Part I,
a browser, a local HTTP server, ESCO, or downloading a transformer model. If Part II's `r` is
available, it is consumed directly. Otherwise the section runs the same `MatchingPipeline`
on a documented synthetic pair already shipped in `data/demo/` using an explicit offline backend.
The resulting numbers check implementation behavior, not formal recommendation accuracy.


In [1]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display, Markdown

# Open the notebook from the project root, or from its immediate parent directory.
learning_root = Path.cwd()
if not (learning_root / "src" / "pipeline.py").exists():
    learning_root = learning_root / "6405-project"
if not (learning_root / "src" / "pipeline.py").exists():
    raise FileNotFoundError("Set the notebook working directory to 6405-project.")
sys.path.insert(0, str(learning_root))
from src.learning import LearningRecommender, gaps_from_match
from src.learning.integration import recommend_from_match
from src.learning.workflow import plan_markdown

learning_output = learning_root / "results" / "learning"
learning_output.mkdir(parents=True, exist_ok=True)
learning_engine = LearningRecommender()
print("Learning corpus:", len(learning_engine.courses), "curated starter units")


Learning corpus: 12 curated starter units


In [2]:
# Reuse the team's real result when available. Keep all original variables unchanged.
LEARNING_USE_EXISTING_RESULT = True
if LEARNING_USE_EXISTING_RESULT and isinstance(globals().get("r"), dict) and "details" in r:
    learning_match_result = r
    learning_input_source = "Part II matching output r (same extraction/matching configuration)"
else:
    from src.pipeline import MatchingPipeline, PipelineConfig
    from src.skill_extraction import DictionarySkillExtractor
    learning_pipeline = MatchingPipeline(
        PipelineConfig(embedder_name="tfidf-char", doc_sim_weight=0.5),
        skill_extractor=DictionarySkillExtractor())
    learning_jd = (learning_root / "data/demo/jds/JD_backend_engineer.txt").read_text(encoding="utf-8")
    learning_resume = (learning_root / "data/demo/resumes/R02_analyst_junior.txt").read_text(encoding="utf-8")
    learning_match_result = learning_pipeline.match(learning_jd, learning_resume)
    learning_input_source = "Synthetic demo pair: backend engineer / junior analyst; offline character n-gram backend"

print("Input:", learning_input_source)
display(learning_match_result["details"])
learning_gaps = gaps_from_match(learning_match_result)
display(pd.DataFrame(learning_gaps))


Input: Synthetic demo pair: backend engineer / junior analyst; offline character n-gram backend


requirement_id,category,requirement,priority,match_score,match_type,resume_evidence,semantic_similarity
R001,Skill,APIs,UNSPECIFIED,0.0,NOT_EVIDENCED,Business Analyst Intern,0.000000
R002,Skill,Microservices,UNSPECIFIED,0.0,NOT_EVIDENCED,Prepared weekly Excel reports and pivot tables for the operations team.,0.013518
R003,Skill,Kubernetes,UNSPECIFIED,0.0,NOT_EVIDENCED,Business Analyst Intern,0.026189
R004,Skill,AWS,UNSPECIFIED,0.0,NOT_EVIDENCED,Business Analyst Intern,0.000000
R005,Experience,Minimum 3 years,REQUIRED,22.3,DURATION_BELOW,0.67 years of dated experience; requires 3.0 years.,NaN
R006,Skill,Java,REQUIRED,0.0,NOT_EVIDENCED,Business Analyst Intern,0.000000
R007,Skill,Go,REQUIRED,0.0,NOT_EVIDENCED,Business Analyst Intern,0.000000
R008,Skill,PostgreSQL,REQUIRED,0.0,NOT_EVIDENCED,SQL,0.235702
R009,Skill,Redis,REQUIRED,0.0,NOT_EVIDENCED,Prepared weekly Excel reports and pivot tables for the operations team.,0.046829
R010,Skill,Docker,REQUIRED,0.0,NOT_EVIDENCED,Business Analyst Intern,0.000000


skill,priority,weight
apis,UNSPECIFIED,1.0
microservices,UNSPECIFIED,1.0
kubernetes,UNSPECIFIED,1.0
aws,UNSPECIFIED,1.0
java,REQUIRED,2.0
go,REQUIRED,2.0
sql,REQUIRED,2.0
redis,REQUIRED,2.0
docker,REQUIRED,2.0
ci/cd,REQUIRED,2.0


### 9.1 Retrieval and coverage-aware selection

The adapter only uses `Skill` rows with score below 50 or unknown score. Education, employment
duration and citizenship constraints are not treated as skills that a course can resolve.
Missing resume evidence is not evidence of missing ability.

For each gap, retrieve by BM25 and TF-IDF, fuse lexical rankings using RRF (`k=60`), then select
resources using priority-weighted new coverage, relevance, redundancy and prerequisite penalties:

`new coverage + 0.15 × normalized relevance − 0.25 × repeated coverage − 0.5 × missing prerequisites`

Priority weights: mandatory=3, required=2, unspecified=1, preferred=0.5.
Only curated source-evidence mappings can count as covered skills. The selected units are ordered
by editorial prerequisite hints. This is a heuristic, not a guarantee of an optimal study plan.


In [3]:
learning_plan = recommend_from_match(learning_match_result, method="hybrid", top_k=5)
learning_table = pd.DataFrame([
    {"step": c["step"], "resource": c["title"], "covers": ", ".join(c["covered_gaps"]),
     "new_topics": ", ".join(c["newly_covered_gaps"]),
     "prerequisites_to_confirm": ", ".join(c["unmet_prerequisites"]), "source": c["url"]}
    for c in learning_plan["recommendations"]])
display(learning_table)
print("Uncovered:", learning_plan["uncovered_skills"])
print("Weighted syllabus-topic coverage:", learning_plan["weighted_coverage"])
display(pd.DataFrame(learning_plan["selection_trace"]))

learning_evidence = pd.DataFrame([
    {"resource_id": c["id"], "skill": skill, "source_span": c["evidence"][skill], "source_url": c["url"]}
    for c in learning_plan["recommendations"] for skill in c["covered_gaps"]])
display(learning_evidence)
learning_table.to_csv(learning_output / "L_recommendations.csv", index=False, encoding="utf-8-sig")
learning_evidence.to_csv(learning_output / "L_source_evidence.csv", index=False, encoding="utf-8-sig")
(learning_output / "L_plan.json").write_text(json.dumps(learning_plan, ensure_ascii=False, indent=2), encoding="utf-8")
(learning_output / "L_plan.md").write_text(plan_markdown(learning_plan), encoding="utf-8")


step,resource,covers,new_topics,prerequisites_to_confirm,source
1,Get started with Docker,docker,docker,,https://docs.docker.com/get-started/
2,Pro Git: Git basics,git,git,,https://git-scm.com/book/en/v2
3,PostgreSQL: the SQL language,sql,sql,,https://www.postgresql.org/docs/current/tutorial-sql.html


Uncovered: ['apis', 'aws', 'ci/cd', 'go', 'java', 'kafka', 'kubernetes', 'microservices', 'redis']
Weighted syllabus-topic coverage: 0.3243


selection_step,resource_id,new_skills,components,objective,eligible_candidates
1,docker-start,[docker],"{'new_coverage': 2.0, 'relevance': 0.15, 'redundancy_penalty': 0.0, 'prerequisite_penalty': 0.0}",2.15000,5
2,git-basics,[git],"{'new_coverage': 2.0, 'relevance': 0.15, 'redundancy_penalty': 0.0, 'prerequisite_penalty': 0.0}",2.15000,4
3,sql-intro,[sql],"{'new_coverage': 2.0, 'relevance': 0.14758, 'redundancy_penalty': 0.0, 'prerequisite_penalty': 0.0}",2.14758,3


resource_id,skill,source_span,source_url
docker-start,docker,Build and share a containerized application,https://docs.docker.com/get-started/
git-basics,git,Git Basics,https://git-scm.com/book/en/v2
sql-intro,sql,The SQL Language,https://www.postgresql.org/docs/current/tutorial-sql.html


In [4]:
# Same gaps and candidate catalog for every configuration; no human accuracy claims.
learning_comparison_rows = []
for learning_method, learning_optimize in [("bm25", False), ("tfidf", False), ("hybrid", False), ("hybrid", True)]:
    learning_comparison_result = learning_engine.recommend(
        learning_gaps, method=learning_method, optimize=learning_optimize, top_k=5)
    learning_comparison_rows.append({
        "method": learning_method, "coverage_optimization": learning_optimize,
        "weighted_topic_coverage": learning_comparison_result["weighted_coverage"],
        "redundancy_ratio": learning_comparison_result["redundancy_ratio"],
        "learning_units": len(learning_comparison_result["recommendations"]),
        "prerequisite_warnings": sum(bool(c["unmet_prerequisites"]) for c in learning_comparison_result["recommendations"])})
learning_comparison = pd.DataFrame(learning_comparison_rows)
display(learning_comparison)
learning_comparison.to_csv(learning_output / "L_pair_ablation.csv", index=False, encoding="utf-8-sig")


method,coverage_optimization,weighted_topic_coverage,redundancy_ratio,learning_units,prerequisite_warnings
bm25,False,0.3243,0.4,5,0
tfidf,False,0.3243,0.4,5,0
hybrid,False,0.3243,0.4,5,0
hybrid,True,0.3243,0.0,3,0


### 9.2 Reproducibility, limitations and contribution evidence

Run `python experiments/evaluate_learning.py --smoke` for the 12 synthetic implementation cases
(48 case/configuration rows). They include aliases, unknown skills, known skills, prerequisites,
priority weighting and redundant resources. These are **not independent judged data**.

Formal retrieval evaluation is prepared in `experiments/evaluate_learning.py`: independently judge
the resource relevance, retain annotator identities, split source profiles by `group_id`, then report
NDCG@K, Recall@K and MRR@K on the held-out split. Unlabelled relevance must not become zero.

Current limits: 12 starter units from 6 providers; manual topic mappings; unknown durations;
lexical retrieval; prerequisite planning can omit useful foundation units. A tiny catalog may give
identical coverage across retrieval methods. Lower redundancy alone does not prove better learning.

The independent contribution is the gap adapter, resource schema, retrieval/fusion, coverage selection,
evidence trace, experimental design and notebook integration. Team matching work remains credited
to the original members. AI assistance and the student's actual validation/modifications should be
declared in the group reflection; this implementation is not evidence of independently hand-written code.


### 9.3 From 12 hand-picked units to a 3,428-course benchmark

The first version of this module could only be checked on 12 author-written cases. To evaluate it like the
rest of the project, we built a benchmark from public data (`src/learning/benchmark.py`):

| Component | Source | Size |
|---|---|---|
| Corpus | Coursera catalogue (title + description) + 12 curated official-doc units | 3,428 resources |
| Skill queries | matcher skill vocabulary with ≥3 tagged courses | 96 |
| Sentence queries | real JD requirement sentences (REQUIRED/PREFERRED labels) naming one of those skills | 37 |
| Gap profiles | real JD (30 labelled) × real resume → skills missing from the resume, with priority | 23 profiles, 4.3 gaps each |
| Silver relevance | Coursera's own skill tags (**never shown to the retrievers**) | 5,642 query–course labels |

Splits are by **skill** (60/20/20): a test skill never appears in training, so the supervised rankers
cannot memorise it. Common-word skills (Excel, Agile, Swift…) must keep their capitalisation, which removed
false queries such as *"You'll excel if you can collaborate"* produced by the dictionary extractor.

In [ ]:
import json, os
from pathlib import Path
import pandas as pd
from IPython.display import Image, display
if "learning_root" in globals():
    os.chdir(learning_root)  # all paths below are relative to the project root
B = Path("results/learning/bench")
print(json.dumps({k: v for k, v in json.load(open("data/learning/benchmark/stats.json")).items() if k != "domains"}, indent=1))
display(pd.read_csv("data/learning/benchmark/queries.csv").groupby(["kind", "split"]).size().unstack())

### 9.4 Retrieval: 12 systems, one per EE6405 topic (`experiments/learning/L1_retrieval_benchmark.py`)

BM25 (term weighting), TF-IDF word / char n-grams (BoW), LSA (SVD feature extraction), LDA (topic model),
GloVe and corpus-trained Word2Vec (word embeddings), RRF and linear fusion (ensembles), and learning-to-rank
with logistic regression / gradient boosting over all of them (supervised ML). With `--neural`, Sentence-BERT
bi-encoders, a cross-encoder re-ranker and our fine-tuned bi-encoder (L4) are added.

**Result (RTX 5060 run, test nDCG@10):** LTR 0.730, fusion + cross-encoder re-rank 0.719, BM25 0.684,
pre-trained Sentence-BERT 0.62–0.63, our fine-tuned bi-encoder 0.619 (best dev score, highest Recall@100 among
dense models, but no gain on unseen test skills with only 75 training queries), GloVe 0.325, LDA 0.164.
Only GloVe and LDA differ significantly from BM25. Word2Vec (0.594) comes from the cloud CPU run
(`results/learning/bench_cloud_cpu/`) because gensim cannot be installed on Python 3.14.
Hyper-parameters (BM25 k1/b, LSA k, LDA k, fusion weights, LTR C) are chosen on the dev split only.
Metrics: nDCG@10 (primary), MAP@100, MRR@10, P@5, Recall@10/100, with 95% bootstrap CIs and a paired
randomisation test against BM25 (Holm-corrected).

In [ ]:
display(pd.read_csv(B / "L1_leaderboard.csv"))
display(Image(B / "fig_L1_retrieval.png", width=650))
display(pd.read_csv(B / "L1_significance.csv"))
s = pd.read_csv(B / "L1_summary.csv")
display(s[s.query_type != "all"].pivot_table(index="system", columns="query_type", values="ndcg@10").round(3)
        .sort_values("skill", ascending=False))
display(pd.read_csv(B / "L1_efficiency.csv"))

### 9.5 From a ranked list to a learning plan (`experiments/learning/L2_plan_evaluation.py`)

A candidate missing *SQL (REQUIRED)* and *Docker (PREFERRED)* does not need five SQL courses. We compare
three ways to build a k-course plan from the gaps: one concatenated query, per-gap queries summed with
priority weights (the original SkillBridge design), and **xQuAD** aspect diversification
(`src/learning/planner.py`). Our xQuAD variant normalises the diversity term by the still-uncovered
priority weight and draws candidates from each gap's own top results; the ablation shows which change matters.
Metric: priority-weighted share of gaps covered by at least one chosen course (silver labels), plus
precision and redundancy.

In [ ]:
display(pd.read_csv(B / "L2_leaderboard_k5.csv"))
display(Image(B / "fig_L2_strategies.png", width=700))
display(pd.read_csv(B / "L2_significance.csv"))
display(pd.read_csv(B / "L2_ablation.csv"))
display(Image(B / "fig_L2_lambda.png", width=520))

### 9.6 Error analysis and label audit (`experiments/learning/L3_error_analysis.py`)

* Buckets: query type (skill name vs JD sentence), query length, number of relevant courses.
* **Silver-label audit:** among the best system's top-10 courses that the silver labels call *non-relevant*,
  we count how many literally mention the skill in their description. These are likely missing tags, so the
  absolute scores are a lower bound; this motivated the human judgements in 9.7.

In [ ]:
eb = pd.read_csv(B / "L3_error_buckets.csv")
display(eb[eb.system.isin(["BM25 (dev-tuned)", "LTR logistic regression (train/dev)", "Word2Vec (trained on corpus)"])]
        .pivot_table(index=["bucket", "bucket_value"], columns="system", values="ndcg@10").round(3))
audit = pd.read_csv(B / "L3_label_audit.csv")
print(f"{len(audit)} top-10 'non-relevant' hits; {audit.mentions_skill.mean():.1%} mention the skill verbatim")
display(audit[audit.mentions_skill].head(8))
display(pd.read_csv(B / "L3_worst_queries.csv").head(6))

### 9.7 Live demo of the benchmark winner and human evaluation

`src/learning/course_search.py` serves dev-tuned BM25 per gap + normalised xQuAD (note:
on the test profiles TF-IDF + xQuAD is slightly higher, 0.921 vs 0.894, see 9.5) over all 3,428 courses; every recommendation shows the description sentence that names the skill.
The same planner is exposed at `POST /api/courses` in `showcase_app.py`.

Human judgements (optional, **not carried out** by the team): `python experiments/learning/L5_human_judgements.py make`
writes two blind, pooled sheets (`data/learning/human/`); `... score` would report Cohen's κ and re-score every
system on human labels. All numbers in this section therefore use the Coursera-derived silver labels.

In [ ]:
from src.learning.course_search import CourseSearch
cs = CourseSearch()
demo_gaps = [{"skill": g["skill"], "priority": g["priority"]} for g in learning_plan["gaps"]] if "learning_plan" in globals() \
    else [{"skill": "SQL", "priority": "REQUIRED"}, {"skill": "Docker", "priority": "PREFERRED"}, {"skill": "Tableau", "priority": "REQUIRED"}]
plan_out = cs.plan(demo_gaps, k=5)
display(pd.DataFrame([{"rank": r["rank"], "course": r["title"], "provider": r["provider"],
                       "covers": ", ".join(r["evidence"]), "evidence": " | ".join(r["evidence"].values())[:160]}
                      for r in plan_out["recommendations"]]))
print("evidence coverage", plan_out["evidence_coverage"], "uncovered", plan_out["uncovered"], f"{plan_out['latency_ms']} ms")
for f in ["L5_agreement.csv", "L5_human_eval.csv"]:
    if (B / f).exists():
        display(pd.read_csv(B / f))

---
## 10. Traditional NLP vs transformers on the fit benchmark (`experiments/step9_classic_vs_neural.py`)

Section 5 compared lexical baselines with transformer matchers. Step 9 fills in the rest of the course's
traditional toolbox on **the same 600 test pairs**: LSA (D9), LDA topic similarity (D10), GloVe and
Word2Vec averaged embeddings (D11), and four supervised fit classifiers on pair features (D12: logistic
regression, linear SVM, multinomial NB, random forest; trained on the 3,000 training pairs). Hyper-parameters
are chosen on a JD-grouped dev split of the training pairs. Optionally (`--finetune-ce`, GPU) a cross-encoder
is fine-tuned with a learning-rate × epoch grid.

**Result (RTX 5060 run):** supervised classifiers on TF-IDF pair features are the best matchers on this
benchmark (Linear SVM ρ = 0.317 [0.188, 0.447], random forest 0.310), ahead of the fine-tuned bi-encoder D7
(0.281) and the newly fine-tuned cross-encoder D13 (0.181, up from −0.075 zero-shot). Unsupervised LSA / LDA /
GloVe and zero-shot transformers are near zero. The logistic-regression weights explain why: its 20 strongest
features are all JD-side words, i.e. the classifiers mostly learn which vacancies tend to be labelled "good fit"
— the same vacancy prior found in Section 5 — so these scores should not be read as person–job fit.

Also reported: efficiency (fit time, ms per pair, peak memory) and interpretation — logistic-regression term
weights, LDA topics that separate good-fit from no-fit resumes, and sentence-occlusion explanations that show
which resume sentences drive each model's score.

In [ ]:
S9 = Path("results/step9")
if not (S9 / "S9_matching_all.csv").exists():
    print("Run: python experiments/step9_classic_vs_neural.py   (needs data/hf from step1)")
else:
    display(pd.read_csv(S9 / "S9_matching_all.csv"))
    display(pd.read_csv(S9 / "S9_classifiers.csv"))
    display(pd.read_csv(S9 / "S9_efficiency.csv"))
    display(pd.read_csv(S9 / "S9_logreg_top_terms.csv"))
    display(pd.read_csv(S9 / "S9_lda_topics.csv").head(8))
    display(pd.read_csv(S9 / "S9_occlusion_examples.csv"))